# Actividad 2A · Tipos de problemas de aprendizaje automático

**Tiempo estimado:** 20 minutos  
**Recorrido:** SQL + Python

En la actividad anterior pasamos de describir datos a formular una pregunta predictiva. Ahora vamos a distinguir **qué tipo de respuesta buscamos**: una categoría, un valor numérico o grupos de casos con características semejantes.

Seguimos trabajando con los datos de clientes del servicio de telecomunicaciones (*churn*). Todavía no vamos a entrenar modelos; primero necesitamos reconocer el problema que queremos resolver.

**Para empezar:** ejecutá la celda de preparación sin modificarla. El notebook descargará los datos y dejará lista la conexión con DuckDB.


## Tres tipos de problemas

| Tipo de problema | Pregunta de ejemplo | Resultado esperado |
|---|---|---|
| Clasificación | ¿El cliente abandonará el servicio? | Una categoría: sí o no |
| Regresión | ¿Qué cargo mensual podría tener un cliente? | Un valor numérico |
| Agrupamiento | ¿Qué perfiles de clientes encontramos? | Grupos de casos similares |

En **clasificación** y **regresión** disponemos de una respuesta conocida en los datos que usamos para aprender a predecir. En **agrupamiento**, en cambio, buscamos similitudes sin indicar de antemano a qué grupo pertenece cada cliente.

Vamos a usar consultas SQL para reconocer estas diferencias. **No estamos construyendo modelos todavía.**


In [ ]:
#@title Preparación automática del entorno (ejecutar sin modificar)
import importlib.util
import subprocess
import sys
from pathlib import Path
from urllib.request import urlretrieve

print("Preparando el entorno...")

for package in ["duckdb", "pandas"]:
    if importlib.util.find_spec(package) is None:
        print(f"Instalando {package}...")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", package
        ])

import duckdb
import pandas as pd

print("Paquetes preparados correctamente.")

DATA_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
DATA_PATH = Path("Telco-Customer-Churn.csv")

if not DATA_PATH.exists():
    print("Descargando dataset...")
    urlretrieve(DATA_URL, DATA_PATH)

churn = pd.read_csv(DATA_PATH)

required_columns = {"Churn", "tenure", "MonthlyCharges", "TotalCharges", "Contract", "InternetService"}
if not required_columns.issubset(churn.columns):
    raise ValueError("El dataset descargado no contiene las columnas esperadas.")

churn["TotalCharges"] = pd.to_numeric(churn["TotalCharges"], errors="coerce")

# Si la celda se ejecuta otra vez, cerramos primero una conexión previa.
if "con" in globals():
    try:
        con.close()
    except Exception:
        pass

con = duckdb.connect(":memory:")
con.register("churn_df", churn)
con.execute("CREATE OR REPLACE TABLE churn AS SELECT * FROM churn_df")

clientes = f"{len(churn):,}".replace(",", ".")
print(f"Entorno listo: {clientes} clientes y {churn.shape[1]} variables.")


## 1. Clasificación · ¿abandona o no abandona?

La variable `Churn` registra si un cliente abandonó el servicio. Con esta consulta vamos a contar cuántos casos corresponden a cada respuesta.

Observá el resultado: ¿la respuesta que querríamos predecir es una categoría o un número?


In [ ]:
clasificacion = con.sql("""
SELECT Churn AS clase, COUNT(*) AS cantidad
FROM churn
GROUP BY Churn
ORDER BY Churn
""").df()
clasificacion


## 2. Regresión · estimar un valor numérico

`MonthlyCharges` representa el cargo mensual del servicio. Consultamos su mínimo, promedio y máximo para reconocer el tipo de valor que podríamos querer estimar.

Estos resúmenes sirven para conocer la variable: **no son, por sí mismos, un modelo de regresión**.


In [ ]:
regresion = con.sql("""
SELECT
    MIN(MonthlyCharges) AS minimo,
    ROUND(AVG(MonthlyCharges), 2) AS promedio,
    MAX(MonthlyCharges) AS maximo
FROM churn
""").df()
regresion


## 3. Agrupamiento · buscar perfiles semejantes

Seleccionamos algunas características de los clientes sin incluir una variable objetivo. Imaginá que queremos identificar perfiles según la antigüedad, el cargo mensual y el tipo de servicio.

La consulta solo muestra los datos disponibles; todavía **no forma grupos**.


In [ ]:
datos_sin_objetivo = con.sql("""
SELECT tenure, MonthlyCharges, Contract, InternetService
FROM churn
LIMIT 8
""").df()
datos_sin_objetivo


## Para comprobar lo trabajado

Leé estas situaciones y decidí qué tipo de problema corresponde a cada una:

1. Predecir si un cliente dará de baja el servicio.
2. Estimar el consumo de datos de un usuario durante el próximo mes.
3. Encontrar perfiles de uso de una aplicación sin definir los grupos previamente.
4. Estimar cuántas horas demandará resolver un incidente de soporte.

Indicá si se trata de **clasificación, regresión o agrupamiento**. Justificá cada respuesta con una frase: ¿buscamos una categoría, un valor numérico o grupos que todavía no conocemos?

**Tus respuestas:** escribilas acá.

## Cierre

El tipo de resultado que buscamos ayuda a decidir qué problema vamos a resolver. En la segunda parte de esta actividad vamos a distinguir el aprendizaje supervisado del no supervisado según la información disponible.


In [ ]:
# Cierre de la conexión con DuckDB
if "con" in globals():
    try:
        con.close()
    except Exception:
        pass
